# BmD cube - Frontend orchestrator of the cubing engine

In previous tutorials we have put the emphasis on the software architecture that is defining for the cubing engine. In this tutorial we will discuss the orchestrator class `bmdCube` which serves as the entrypoint for users to request and process data as is required for them.

The `bmdCube` can be seen as an orchestrator of the data pipeline due to the fact that it takes the requested recipe and creates an instance of all dataset cubes such as `chelsaCube` or `gbifCube` and parses the recipe so that they fetch and process data as the user request. This allows the user to focus solely on understanding the recipe and the associated syntax to communicate what the end product of the cube should look like.

In this tutorial we will focus on the recipe structure and showcase how to utilize the produced cube in an efficient way

## Recipe structure & syntax

### General parameters

The first component of the recipe can be described as a general set of parameters that determine
- where the end product of the pipeline is written to
- what the directory name containing the cube will be called
- what the file format for the raster type data
- custom execution configuration


```yaml
base_dir: "./test_outputs/"
cube_name: "dev_cube"

# Only include this if you want to change the execution configuration 
# For more explanation of these options, consult the readme file in the config directory
"""
execution:
    min_slices_for_parallel: 16,
    max_safe_threadpool_workers: 8,
    large_slice_threshold_mb: 50.0,
    gdal_warp_block_size: 2048,
    warp_cache_cleanup_delay_sec: 30
"""

export_as:
  format: "zarr"
```

The combination of `base_dir` and `cube_name` will form the output directory to which all the results are written to. In addition to this we allow for custom execution parameters to be defined within the recipe. These optional parameters are commented out in the template as standard users should not change to much in these settings, experienced developers can tweak these with respect to the computational infrastructure that is available to them. 

### Spatial & temporal parameters

```yaml
# ======================================================================
# 1. SPATIAL & TEMPORAL SELECTION
# ======================================================================
spatial:
  target_grid: "EEA"
  target_resolution: "1km"
  use_bbox: true  
  bbox: #Flanders bbox
      long_min: 2.591733
      long_max: 5.883999
      lat_min: 50.680797
      lat_max: 51.963900
temporal:
  start_year: 2004  #start year of the european invasion of the asian hornet
  start_month: 1
  end_year: 2025
  end_month: 12
  time_aggregation: "monthly"
```

The spatial and temporal parameters define the extent to which the requested datasets are queried and processed. For the cubing engine we have access to different grids to which the data can be harmonized

```python
Available Master Grids:
 - EEA_10m: EPSG:3035 | Res: 10
 - EEA_100m: EPSG:3035 | Res: 100
 - EEA_250m: EPSG:3035 | Res: 250
 - EEA_500m: EPSG:3035 | Res: 500
 - EEA_1km: EPSG:3035 | Res: 1000
 - EEA_10km: EPSG:3035 | Res: 10000
 - Global_EqualArea_10m: EPSG:6933 | Res: 10
 - Global_EqualArea_100m: EPSG:6933 | Res: 100
 - Global_EqualArea_250m: EPSG:6933 | Res: 250
 - Global_EqualArea_500m: EPSG:6933 | Res: 500
 - Global_EqualArea_1km: EPSG:6933 | Res: 1000
 - Global_EqualArea_10km: EPSG:6933 | Res: 10000
 - Global_WGS84_0_3sec: EPSG:4326 | Res: 8.333333333333333e-05
 - Global_WGS84_3sec: EPSG:4326 | Res: 0.0008333333333333333
 - Global_WGS84_7_5sec: EPSG:4326 | Res: 0.0020833333333333333
 - Global_WGS84_15sec: EPSG:4326 | Res: 0.004166666666666667
 - Global_WGS84_30sec: EPSG:4326 | Res: 0.008333333333333333
 - Global_WGS84_5min: EPSG:4326 | Res: 0.08333333333333333
```

When constructing the recipe you provide either `EEA`, `Global_EqualArea` or `Global_WGS84` in the `target_grid` section. For the `resolution` section the options are `10m`, `100m`, `250m`, `500m`, `1km`, `10km`, `0_3sec`, `3sec`, `7_5sec`, `15sec`, `30sec` or `5min`.

For the temporal parameters we operate on closed intervals, i.e. data will be sampled within `[start_year, end_year]` and `[start_month, end_month]`

### Datasource section

#### Generic datasource

For simple datasources such as `CHELSA` the recipe serves as a sort of switchboard where the user can toggle which variables are to be included in the produced cube. If we use `chelsa` as an example 

```yaml
# --------------------------------------------------------------------
  # CHELSA DATASETS (High-Resolution Climatologies)
  # --------------------------------------------------------------------
  chelsa:
    enabled: true
    catalog_path:  ""
    levels: 
      # --- DAILY TIME SERIES (Inherits global 'temporal' block) ---
      daily:
        include: true
        variables:
          tas: true     # Mean daily air temperature
          tasmax: false # Maximum daily air temperature
          tasmin: false # Minimum daily air temperature
          pr: false     # Daily precipitation amount
          clt: false    # Cloud area fraction
          rsds: false   # Surface downwelling shortwave radiation
          sfcWind: false # Near-surface wind speed
          hurs: false   # Near-surface relative humidity
          vpd: false    # Vapor pressure deficit
```

We see that in order to include it we set the `enabled` boolean variable to true followed by the levels we wish to include (for CHELSA these levels are `daily`, `monthly`, `annual`, `bioclim`, `climatologies`). In the library an example of all options is provided within the `template_recipe.yaml` file located in the `config` folder in the base directory. Variables or levels that **are not of interest to the user do not need to be included explicitly in the recipe (these are shown here and in the template explicitly to highlight the reasoning of the recipe syntax**. A catalog is required for the engine to properly function, this is a file that follows the structure outlined in the `2a_rasterCube.ipynb` and `2b_vectorCube.ipynb` tutorials meant for backend developers. Users can provide their own catalog to the recipe however, within the dataset adaptor classes the development team has incorporated standard catalogs for each dataset stored in the `meta` directory in the base folder. As such, users do not need to be concerned with this specific option and can level it as an empty string 

#### GBIF

GBIF represents a more complex recipe syntax due to the fact that we offer interfacing with the GBIF SQL API required for downloading live data from their servers.

```yaml
gbif:
    #When the user wishes to load in a file on disc provide path here
    local_file_path: 
    # =======================================================================
    # 1. QUERY & FILTER CONFIGURATION
    # Defines exactly what data is pulled from the GBIF network.
    # =======================================================================
    query_filters:
      # General taxonKey parser; accepts keys from all taxonomic levels (e.g., family, genus, species).
      taxon_keys: ['G59D','4F6YZ','3Y9W2','5B9T3','4HPXM','3Y9VW','75SVV','3ZW55','3ZDRX','6CJDB','GGZL','6JYHZ','7G3C6']
      
      # If true, ignores taxon_keys and fetches ALL occurrence data within the Area of Interest (AOI).
      fetch_all_taxa: false
      
      # Determines the type of records to fetch. Options: "presence", "absence", or "mixed" (both).
      record_type: "presence"
      
      # Fallback coordinate uncertainty (in meters) assigned to records missing this value.
      # If left blank, defaults to matching the spatial target_resolution.
      default_Uncertainty: 1000
      
      # Hard cutoff for coordinate uncertainty. Records exceeding this radius (in meters) are dropped.
      max_uncertainty: 1000
      
      # GBIF data quality flags to exclude. Drops records flagged with these specific issues.
      exclude_issues:
        - "ZERO_COORDINATE"
        - "COORDINATE_OUT_OF_RANGE"
        - "COUNTRY_COORDINATE_MISMATCH"
    
    # =======================================================================
    # 2. TAXONOMIC HARMONIZATION
    # Controls how taxonomic lineages are resolved and appended.
    # =======================================================================
    taxonomy:
      # If true, cross-references and aligns taxa against the Catalogue of Life (CoL) backbone.
      col_backbone: true
      
      # Defines the deepest taxonomic rank to include (kingdom, phylum, class, order, family, genus).
      # If left empty, only the species key will be retained as an identifier in the final table.
      lowest_level: 

    # =======================================================================
    # 3. ATTRIBUTE RETENTION
    # =======================================================================
    # Additional Darwin Core columns to extract and retain alongside the geometries.
    columns: ["year", "month"]
    time_cols: ["year", "month"] #Information for the CF temporal conversion
```

Based on the entries that the user provides in the recipe the `bmc` library constructs a SQL query that can be submitted to the GBIF server to retrieve all required data. If the user wishes to skip the download using a GBIF SQL query they can alternatively provide a darwincore csv table stored at a specified location which can be fed to the cubing engine on the `local_file_path` field. Explanation for each field is provided in the template recipe as shown above.

Processing the GBIF data requires the specification of a `vector_processing` field that is similarly structured for different datasources

#### `vector_processing`

When handling vector data the user can make specific requests how the data should be processed and aggregated in the resulting cube

```yaml
# =======================================================================
    # 4. SPATIAL INGESTION & PROCESSING ROUTING
    # =======================================================================
    # Defines exactly how the pipeline handles spatial geometries.
    # Options: 
    #  - "raw": Bypasses all grid math. Returns raw, unmapped occurrence records.
    #  - "vector": Downloads raw points, applies geometric topologies (buffers/clouds), and maps to target grid.
    #  - "api_cube": Uses GBIF's server-side pre-aggregated SQL grid API (fast, contracted data).
    processing_mode: "vector"

    # -----------------------------------------------------------------------
    # ROUTE A: PURE VECTOR PROCESSING (Used when processing_mode: "vector")
    # Handles raw coordinate mathematics and spatial transformations.
    # -----------------------------------------------------------------------
    vector_processing:
      # Desired geometric topology: "point", "polygon", or "point_cloud".
      topology: "point_cloud"
      
      # Mapping output: "classification" (1-to-1 cell assignment) or 
      # "fractional" (1-to-N probability/areal weights).
      mapping_mode: "fractional"  
      
      # Spatial lookup strategy: "intersect" (precise overlap) or "kdtree" (nearest centroid).
      # Note: KDTree is only mathematically valid when mapping_mode is 'classification'.
      spatial_method: "intersect"

      # Topology-specific configurations passed directly to the geometry engines.
      # The orchestrator will only apply the config matching the active 'topology'.
      topology_config:
        point_cloud:
          # Number of randomized points generated in the point cloud
          n_passes: 100
          
          # Underlying probability distribution (either "gaussian" or "uniform") 
          # with a width determined by the uncertainty radius
          distribution: "gaussian"
          
          # Random seed generator to ensure reproducibility
          random_seed: 42

        polygon:
          # Number of quadrant segments of the polygon determining the smoothness of the polygon being constructed.
          # 2-4 is very coarse, 16-32 is very smooth, 8 is the default value in many libraries.
          # Note: Increasing this will increase the impact on RAM memory during intersection operations.
          quad_segs: 8

    # -----------------------------------------------------------------------
    # ROUTE B: GBIF API CUBE CONFIGURATION (Used when processing_mode: "api_cube")
    # Maps pre-aggregated GBIF SQL grid cells using cell collection routines.
    # -----------------------------------------------------------------------
    api_cube_config:
      # "discrete" (integer sums/majority rule) or "continuous" (areal weighting).
      data_type: "discrete"       
      
      # "intersect" (geometric overlap/clipping) or "kdtree" (centroid snapping).
      spatial_method: "intersect"

    # =======================================================================
    # 5. DATA AGGREGATION & METRICS
    # =======================================================================
    aggregate:
      export_unaggregated: true
      
      # Defines the non-spatial dimensions of the final data cube.
      # The spatial cell ID (grid_idx) is automatically appended by the engine.
      group_by_columns: ["year", "month"]
      
      metrics:
        # 1. Expected Occurrences (Sums the geometric fractions)
        - column: "areal_fraction"
          method: "sum"
          weighted: false
          rename: "expected_occurrences"
          
        # 2. Species Count / Richness
        - column: "speciesKey"
          method: "nunique"
          weighted: false
          rename: "species_richness"
          
        # 3. Distinct Observers
        - column: "recordedBy"
          method: "nunique"
          weighted: false
          rename: "distinct_observers"
```

## Generating a test cube

## Using the cubing engines output 

### General structure of the engine output

The general output of the cubing engine is a nested directory 

<img src="../img/datatreeDark.png" width="1000">

The engine writes all the files to a directory `base_dir/cube_name` and the general structure is 

1) A collection of folder named after the requested datasets containing the files that describe the requested variables
- For raster files this will be a set of `netcdf` or `zarr` files
- For vector data this will be a set of parquet files
2) A meta directory containing a 
- STAC directory with the STAC entry of the generated cube
- provenance json record that fingerprints the utilized software and hardware
3) A log directory containing
- log file for each dataset being processed
- a resource log file that profiles the code